# Custom CNN Training Sanity Check

This notebook uses the project's existing data loader, `CustomCNN`, and training utilities to verify the end-to-end training path. It runs one training epoch and one validation pass only; it is not a long training experiment.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision.utils import make_grid

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## 2. Connect the project source

In [ ]:
project_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    candidate for candidate in project_candidates
    if (candidate / "src" / "image_classifier").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from image_classifier.config import BATCH_SIZE, DEVICE, LEARNING_RATE
from image_classifier.data_loader import create_dataloaders
from image_classifier.models.cnn import CustomCNN
from image_classifier.training.train import train_one_epoch, validate_one_epoch

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Source directory added to sys.path: {SRC_DIR.resolve()}")

## 3. Load the data

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders()

print(f"Training samples: {len(train_loader.dataset):,}")
print(f"Validation samples: {len(val_loader.dataset):,}")
print(f"Test samples: {len(test_loader.dataset):,}")
print(f"Training batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")
print(f"Test batches: {len(test_loader):,}")
print(f"Batch size: {train_loader.batch_size} (configured value: {BATCH_SIZE})")

## 4. Inspect a training batch

In [ ]:
images, labels = next(iter(train_loader))

print(f"Image tensor shape: {tuple(images.shape)}")
print(f"Label tensor shape: {tuple(labels.shape)}")
print(f"First several labels: {labels[:8].tolist()}")
print(f"Image tensor dtype: {images.dtype}")
print(f"Label tensor dtype: {labels.dtype}")

## 5. Visualize normalized training images

In [ ]:
imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

images_to_show = images[:8].cpu()
labels_to_show = labels[:8].cpu()
denormalized_images = (images_to_show * imagenet_std + imagenet_mean).clamp(0, 1)
grid = make_grid(denormalized_images, nrow=4, padding=2)

fig, axis = plt.subplots(figsize=(10, 5.5))
axis.imshow(grid.permute(1, 2, 0))
axis.set_title("Training Batch Samples (ImageNet Normalization Reversed)")
axis.set_xlabel("Labels: " + ", ".join(str(label.item()) for label in labels_to_show))
axis.axis("off")
fig.tight_layout()
plt.show()

## 6. Create and inspect the CustomCNN

In [ ]:
model = CustomCNN(num_classes=6)
print(model)

In [ ]:
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")

## 7. Forward pass and initial loss

In [ ]:
loss_fn = nn.CrossEntropyLoss()

with torch.no_grad():
    initial_outputs = model(images)
    initial_loss = loss_fn(initial_outputs, labels)

print(f"Input tensor shape: {tuple(images.shape)}")
print(f"Output tensor shape: {tuple(initial_outputs.shape)}")
print(f"CrossEntropyLoss for the batch: {initial_loss.item():.4f}")

## 8. Training configuration

In [ ]:
if DEVICE == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(DEVICE)

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Device: {device}")

## 9. One training epoch and validation

In [ ]:
train_loss, train_accuracy = train_one_epoch(
    model=model,
    dataloader=train_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
)

validation_loss, validation_accuracy = validate_one_epoch(
    model=model,
    dataloader=val_loader,
    loss_fn=loss_fn,
    device=device,
)

print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.2%}")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {validation_accuracy:.2%}")

## Interpretation

This one-epoch run is an executable pipeline check rather than a performance benchmark. The training and validation metrics show that the model, data transforms, loss function, optimizer, and device placement work together; meaningful generalization conclusions require a planned multi-epoch experiment.